# Paper 1 schema probe 3: TPEx date parameter (not a test)
Requests only the burned dates 2024-01-02 and 2024-01-03. A response is kept only if its own date field equals the requested date; otherwise only its date and hash are logged and the body is discarded. Writes `schema_probe3_results.zip`.

Everything is in **one cell**: run it once.

In [ ]:
import io, json, re, time, zipfile, datetime, hashlib, requests
OUT, LOG = {}, []
H = {"User-Agent": "Mozilla/5.0 (research schema probe)", "Accept": "application/json"}
WANT = {"20240102": ("2024/01/02", "113/01/02"), "20240103": ("2024/01/03", "113/01/03")}
def resp_date(j):
    for k in ("date", "reportDate"):
        if k in j: return str(j[k])
    t = (j.get("tables") or [{}])[0]
    return str(t.get("date", ""))
def norm(s):
    s = re.sub(r"[^0-9]", "", s)
    if len(s) == 7: s = str(int(s[:3]) + 1911) + s[3:]
    return s
def try_req(name, method, url, params, want):
    t = datetime.datetime.utcnow().isoformat() + "Z"
    try:
        r = requests.request(method, url, headers=H, params=params if method == "GET" else None,
                             data=params if method == "POST" else None, timeout=60)
        h = hashlib.sha256(r.content).hexdigest()
        try: j = r.json(); d = resp_date(j)
        except Exception: j, d = None, ""
        ok = r.ok and norm(d) == want
        rec = {"name": name, "method": method, "url": url, "params": params, "status": r.status_code,
               "bytes": len(r.content), "sha256": h, "returned_date": d, "date_matches": ok, "utc": t}
        if ok:
            OUT[name + ".raw"] = r.content
            tb = (j.get("tables") or [{}])[0]
            rec["fields"] = tb.get("fields") or j.get("fields")
            rec["rows"] = len(tb.get("data", j.get("aaData", [])) or [])
        LOG.append(rec)
    except Exception as e:
        LOG.append({"name": name, "url": url, "params": params, "error": repr(e), "utc": t})
    time.sleep(3)

for want, (greg, roc) in WANT.items():
    variants = [
        ("legacy_quotes_se", "GET", "https://www.tpex.org.tw/web/stock/aftertrading/daily_close_quotes/stk_quote_result.php",
         {"l": "zh-tw", "o": "json", "d": roc, "se": "EW"}),
        ("new_dailyQ_get", "GET", "https://www.tpex.org.tw/www/zh-tw/afterTrading/dailyQ",
         {"date": greg, "type": "EW", "response": "json"}),
        ("new_dailyQ_post", "POST", "https://www.tpex.org.tw/www/zh-tw/afterTrading/dailyQ",
         {"date": greg, "type": "EW", "response": "json"}),
        ("new_otc_get", "GET", "https://www.tpex.org.tw/www/zh-tw/afterTrading/otc",
         {"date": greg, "type": "EW", "response": "json"}),
        ("new_insti_get", "GET", "https://www.tpex.org.tw/www/zh-tw/insti/dailyTrade",
         {"type": "Daily", "sect": "EW", "date": greg, "response": "json"}),
    ]
    for tag, m, url, p in variants:
        try_req(f"tpex_{tag}_{want}", m, url, p, want)

OUT["log.json"] = json.dumps(LOG, indent=1, ensure_ascii=False).encode()
with zipfile.ZipFile("schema_probe3_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for k, v in OUT.items(): z.writestr(k, v)
for e in LOG: print(e["name"], e.get("status"), e.get("returned_date"), e.get("date_matches"), e.get("rows"), e.get("error", ""))
try:
    from google.colab import files; files.download("schema_probe3_results.zip")
except Exception: print("Saved schema_probe3_results.zip")